# Demo 2 — MLflow RAG Evaluation: llama3.2 vs mistral

Demonstrates `mlflow.evaluate()` for RAG quality metrics, comparing two Ollama models
routed through the MLflow AI Gateway. Results are logged to the MLflow Tracking Server
and viewable in the UI.

| Metric | What it measures |
|---|---|
| **faithfulness** | Is the answer grounded in the retrieved context? |
| **answer_relevance** | Does the answer address the question? |
| **context_relevance** | Is the retrieved context relevant to the question? |

**Judge LLM:** mistral (7B) via Ollama  
**Answer generators:** `llama-chat` (llama3.2 3B) vs `mistral-chat` (mistral 7B)

| Service | URL |
|---|---|
| **MLflow Tracking UI** | http://localhost:5001 |
| **Gateway** | http://localhost:5050 |

**Prerequisites:** `docker compose up -d` in `demo2-mlflow-gateway/` — all services healthy.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

GATEWAY_URI      = os.getenv("MLFLOW_GATEWAY_URI",   "http://localhost:5050")
OLLAMA_BASE_URL  = os.getenv("OLLAMA_BASE_URL",       "http://localhost:11434")
TRACKING_URI     = os.getenv("MLFLOW_TRACKING_URI",   "http://localhost:5001")

print(f"Gateway      : {GATEWAY_URI}")
print(f"Ollama       : {OLLAMA_BASE_URL}")
print(f"MLflow UI    : {TRACKING_URI}")

## 1. Build the RAG Evaluation Dataset

Each row: question + retrieved context + generated answer + ground truth.
We generate answers via the gateway, then score them with `mlflow.evaluate()`.

In [ ]:
import time
import pandas as pd
from mlflow.deployments import get_deploy_client

client = get_deploy_client(GATEWAY_URI)

eval_data = [
    {
        "question": "What is the standard deduction for a single filer in 2024?",
        "context": "For tax year 2024, the standard deduction amounts are: Single or Married Filing Separately: $14,600. Married Filing Jointly or Qualifying Surviving Spouse: $29,200. Head of Household: $21,900.",
        "ground_truth": "The standard deduction for a single filer in 2024 is $14,600.",
    },
    {
        "question": "What is the long-term capital gains rate for a single filer earning $100,000?",
        "context": "2024 Long-Term Capital Gains Tax Rates: 0%: Taxable income up to $47,025 (single). 15%: Taxable income from $47,026 to $518,900 (single). 20%: Taxable income above $518,900 (single).",
        "ground_truth": "A single filer earning $100,000 pays 15% on long-term capital gains.",
    },
    {
        "question": "What is the 401(k) employee contribution limit for 2024?",
        "context": "401(k), 403(b), AND 457(b) PLANS: Employee Contribution Limit: $23,000. Catch-Up Contribution (age 50 or older): Additional $7,500 (total $30,500).",
        "ground_truth": "The 401(k) employee contribution limit for 2024 is $23,000, or $30,500 with catch-up contributions for those 50 and older.",
    },
    {
        "question": "What is the Roth IRA income phase-out range for single filers in 2024?",
        "context": "ROTH IRA: Income phase-out ranges for contributions: Single/Head of Household: $146,000 – $161,000 MAGI.",
        "ground_truth": "The Roth IRA income phase-out range for single filers in 2024 is $146,000 to $161,000 MAGI.",
    },
    {
        "question": "Can I deduct mortgage interest on a second home?",
        "context": "Mortgage Interest: Interest on loans secured by your primary or secondary residence is deductible on up to $750,000 of acquisition debt (for loans originated after December 15, 2017).",
        "ground_truth": "Yes, mortgage interest on a secondary residence is deductible on up to $750,000 of acquisition debt for loans originated after December 15, 2017.",
    },
]

# Generate answers via llama-chat
print("Generating answers via llama-chat (llama3.2 3B)...")
llama_rows = []
for row in eval_data:
    prompt = f"Answer the following tax question based only on the provided context.\n\nContext: {row['context']}\n\nQuestion: {row['question']}\n\nAnswer concisely."
    t0 = time.time()
    resp = client.predict(endpoint="llama-chat", inputs={"messages": [{"role": "user", "content": prompt}]})
    answer = resp["choices"][0]["message"]["content"]
    llama_rows.append({**row, "answer": answer, "latency_s": round(time.time() - t0, 2)})
    print(f"  [{llama_rows[-1]['latency_s']:.1f}s] {row['question'][:60]}")

llama_df = pd.DataFrame(llama_rows)
print(f"\nDone. Mean latency: {llama_df['latency_s'].mean():.2f}s")

## 2. Set Up MLflow Experiment

Log to the MLflow Tracking Server so we can compare runs in the UI.

In [ ]:
import mlflow

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("tax-rag-evaluation")

print(f"Tracking URI : {mlflow.get_tracking_uri()}")
print(f"Experiment   : tax-rag-evaluation")
print(f"MLflow UI    : {TRACKING_URI}")

## 3. Evaluate llama3.2 Answers (mistral as Judge)

`mlflow.evaluate()` calls the judge LLM to score each row on faithfulness, answer relevance,
and context relevance. We use **mistral** (7B) as the judge via Ollama's OpenAI-compatible endpoint.

In [ ]:
from mlflow.metrics.genai import faithfulness, answer_relevance, relevance

OLLAMA_V1   = f"{OLLAMA_BASE_URL}/v1"
JUDGE_MODEL = "mistral"

with mlflow.start_run(run_name="llama32-answers-mistral-judge") as run:
    mlflow.log_param("rag_model",     "llama3.2")
    mlflow.log_param("judge_model",   JUDGE_MODEL)
    mlflow.log_param("gateway_route", "llama-chat")
    mlflow.log_param("num_questions", len(llama_df))
    mlflow.log_metric("mean_latency_s", llama_df["latency_s"].mean())
    mlflow.log_metric("max_latency_s",  llama_df["latency_s"].max())

    llama_results = mlflow.evaluate(
        data=llama_df[["question", "context", "answer", "ground_truth"]],
        targets="ground_truth",
        predictions="answer",
        model_type="question-answering",
        extra_metrics=[
            faithfulness(    model=f"openai:/{JUDGE_MODEL}"),
            answer_relevance(model=f"openai:/{JUDGE_MODEL}"),
            relevance(       model=f"openai:/{JUDGE_MODEL}"),
        ],
        evaluator_config={
            "col_mapping": {"inputs": "question", "context": "context"},
            "openai_api_key":  "ollama",
            "openai_api_base": OLLAMA_V1,
        },
    )
    llama_run_id = run.info.run_id

print(f"Run ID: {llama_run_id}")
print("\nAggregate metrics (llama3.2):")
for k, v in llama_results.metrics.items():
    print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")

## 4. Generate mistral Answers and Evaluate

Swap `llama-chat` → `mistral-chat`. Same eval harness, same judge.
This is how the gateway enables apples-to-apples model comparisons.

In [ ]:
print("Generating answers via mistral-chat (mistral 7B)...")
mistral_rows = []
for row in eval_data:
    prompt = f"Answer the following tax question based only on the provided context.\n\nContext: {row['context']}\n\nQuestion: {row['question']}\n\nAnswer concisely."
    t0 = time.time()
    try:
        resp = client.predict(endpoint="mistral-chat", inputs={"messages": [{"role": "user", "content": prompt}]})
        answer = resp["choices"][0]["message"]["content"]
    except Exception as e:
        answer = f"ERROR: {e}"
    elapsed = time.time() - t0
    mistral_rows.append({**row, "answer": answer, "latency_s": round(elapsed, 2)})
    print(f"  [{elapsed:.1f}s] {row['question'][:60]}")

mistral_df = pd.DataFrame(mistral_rows)
print(f"\nDone. Mean latency: {mistral_df['latency_s'].mean():.2f}s")

In [ ]:
with mlflow.start_run(run_name="mistral-answers-mistral-judge") as run:
    mlflow.log_param("rag_model",     "mistral")
    mlflow.log_param("judge_model",   JUDGE_MODEL)
    mlflow.log_param("gateway_route", "mistral-chat")
    mlflow.log_param("num_questions", len(mistral_df))
    mlflow.log_metric("mean_latency_s", mistral_df["latency_s"].mean())
    mlflow.log_metric("max_latency_s",  mistral_df["latency_s"].max())

    mistral_results = mlflow.evaluate(
        data=mistral_df[["question", "context", "answer", "ground_truth"]],
        targets="ground_truth",
        predictions="answer",
        model_type="question-answering",
        extra_metrics=[
            faithfulness(    model=f"openai:/{JUDGE_MODEL}"),
            answer_relevance(model=f"openai:/{JUDGE_MODEL}"),
            relevance(       model=f"openai:/{JUDGE_MODEL}"),
        ],
        evaluator_config={
            "col_mapping": {"inputs": "question", "context": "context"},
            "openai_api_key":  "ollama",
            "openai_api_base": OLLAMA_V1,
        },
    )
    mistral_run_id = run.info.run_id

print(f"Run ID: {mistral_run_id}")
print("\nAggregate metrics (mistral):")
for k, v in mistral_results.metrics.items():
    print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")

## 5. Per-Row Results Comparison

In [ ]:
llama_per_row   = llama_results.tables["eval_results_table"]
mistral_per_row = mistral_results.tables["eval_results_table"]

comparison = pd.DataFrame({
    "question":                 pd.Series([r["question"] for r in eval_data]),
    "llama32_faithfulness":     llama_per_row.get("faithfulness/v1/score"),
    "mistral_faithfulness":     mistral_per_row.get("faithfulness/v1/score"),
    "llama32_answer_relevance": llama_per_row.get("answer_relevance/v1/score"),
    "mistral_answer_relevance": mistral_per_row.get("answer_relevance/v1/score"),
})

print("=== llama3.2 (3B) vs mistral (7B) RAG Quality ===")
comparison

## 6. Summary: Side-by-Side Metric Comparison

In [ ]:
print(f"{'Metric':30s}  {'llama3.2':>10s}  {'mistral':>10s}  {'diff':>10s}")
print("-" * 65)
for key in ["faithfulness/v1/mean", "answer_relevance/v1/mean", "relevance/v1/mean"]:
    l = llama_results.metrics.get(key, 0)
    m = mistral_results.metrics.get(key, 0)
    label = key.split("/")[0]
    print(f"{label:30s}  {l:10.3f}  {m:10.3f}  {m-l:+10.3f}")

print("-" * 65)
print(f"{'mean_latency_s':30s}  {llama_df['latency_s'].mean():10.2f}  {mistral_df['latency_s'].mean():10.2f}")

## 7. Open the MLflow Tracking UI

All runs were logged to the MLflow Tracking Server. Open the UI to explore:
- **Experiment view** — both runs side-by-side
- **Metric comparison** — bar/scatter charts across faithfulness, answer_relevance, relevance
- **Run detail** — per-row eval table with judge explanations
- **Parameter diff** — which model, route, judge was used

In [ ]:
print(f"""
MLflow Tracking UI  →  {TRACKING_URI}

Navigate to:
  Experiments → tax-rag-evaluation
  Select both runs → Compare

Run IDs:
  llama3.2 : {llama_run_id}
  mistral  : {mistral_run_id}
""")

# Open in browser (Mac/Linux)
import subprocess
try:
    subprocess.run(["open", TRACKING_URI], check=False)
    print("Opened in browser.")
except FileNotFoundError:
    print(f"Open manually: {TRACKING_URI}")